In [1]:
import os
import time
import string

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates

from tqdm import tqdm

# Font
from matplotlib import font_manager
font_path = "/workspace/fonts/"
font_list = os.listdir(font_path)
for font_file in font_list:
    try:
        font_manager.fontManager.addfont(font_path + font_file)
    except:
        raise Exception(f"Cannot Load {font_path+font_file}")

plt.rcParams["figure.dpi"] = 300

# Handling the DB

In [2]:
import pymysql

from KISTI_DB_Manager import manage, preview
import importlib as imp
imp.reload(manage), imp.reload(preview)

(<module 'KISTI_DB_Manager.manage' from '/workspace/1.1.KISTI_DB_Manager/KISTI_DB_Manager/manage.py'>,
 <module 'KISTI_DB_Manager.preview' from '/workspace/1.1.KISTI_DB_Manager/KISTI_DB_Manager/preview.py'>)

# NTIS

In [3]:
path = '../Data/Funding/KR_NTIS/'
sorted([x for x in os.listdir(path) if x[0] != '.'])

['1_Projects.ftr',
 '1_Projects_Desc.csv',
 '2_Patents.ftr',
 '2_Patents_Desc.csv',
 '3_Papers.ftr',
 '3_Papers_Desc.csv',
 '4.0_Outputs_생물자원.ftr',
 '4.0_Outputs_생물자원_Desc.csv',
 '4.1_Outputs_생명정보.ftr',
 '4.1_Outputs_생명정보_Desc.csv',
 '4.2_Outputs_신품종.ftr',
 '4.2_Outputs_신품종_Desc.csv',
 '4.3_Outputs_화합물.ftr',
 '4.3_Outputs_화합물_Desc.csv',
 '4.4_Outputs_소프트웨어.ftr',
 '4.4_Outputs_소프트웨어_Desc.csv',
 '4.5_Outputs_연구보고서.ftr',
 '4.5_Outputs_연구보고서_Desc.csv',
 '4.6_Outputs_기술요약정보.ftr',
 '4.6_Outputs_기술요약정보_Desc.csv',
 '4.7_Outputs_연구시설장비.ftr',
 '4.7_Outputs_연구시설장비_Desc.csv',
 '5_CoConsignments.ftr',
 '5_CoConsignments_Desc.csv',
 '6_TechFee.ftr',
 '6_TechFee_Desc.csv',
 '7_Commercialize.ftr',
 '7_Commercialize_Desc.csv',
 '8_TraningSupports.ftr',
 '8_TraningSupports_Desc.csv',
 '9_Training.ftr',
 '9_Training_Desc.csv',
 'Raw']

In [5]:
PATH = '../Data/Funding/KR_NTIS/'
SEP = '\t'
Port = 0 # Port for DB with host
CHARACTER_SET = 'utf8mb4'
COLLATE = 'utf8mb4_unicode_520_ci'

params = dict(Extra_ratio=1.5, 
              Min_Year=1900, 
              Max_Year=2100, 
              unique_ratio_th=.5, 
              freq_ratio_th=1e-3)

db_config = {
    'host': 'localhost',  # Update as needed
    'user': 'user',       # Update as needed
    'password': '1234',       # Update as needed
    'database': 'KR_NTIS_20240429_raw'  # Update as needed
}

data_config = {
    'PATH': PATH,
    'SEP': SEP,
    'file_name': 'file_name', # Dummy init value
    'file_type': 'csv', # Dummy init value
    'table_name': 'table_name', # Dummy init value for Exporting
    'out_path': '../Data/SQL/', # Update as needed
    'Conv_DATETIME': False,
}

In [ ]:
# manage.init_MySQL()
# try:
#     manage.drop_DB(db_config['database'], db_config)
# except:
#     pass
# manage.create_DB(db_config['database'], CHARACTER_SET, COLLATE, db_config)


# # Generate the Tabular File list
# flist = sorted([x for x in os.listdir(PATH) if '.ftr' in x])
# for f in tqdm(flist[:]):
#     data_config = preview.update_data_config(f, data_config)
#     df_desc = preview.get_Table_Description(data_config, params)
    
#     # Generate and execute CREATE TABLE SQL
#     manage.create_table(data_config, db_config)
#     manage.fill_table_from_file(data_config, db_config)
#     manage.set_index(db_config, data_config)
#     manage.optimize_table(db_config, data_config)

 * Starting MySQL database server mysqld
   ...done.
Database `KR_NTIS_20240429_raw` dropped successfully.
Database `KR_NTIS_20240429_raw` created successfully.


  0%|                                                    | 0/16 [00:00<?, ?it/s]

In [7]:
manage.init_MySQL()
try:
    manage.drop_DB(db_config['database'], db_config)
except:
    pass
manage.create_DB(db_config['database'], CHARACTER_SET, COLLATE, db_config)


# Generate the Tabular File list
flist = sorted([x for x in os.listdir(PATH) if '.ftr' in x])
df_descs = []
for f in tqdm(flist[:]):
    data_config = preview.update_data_config(f, data_config)
    df_desc = preview.get_Table_Description(data_config, params)
    df_descs.append(df_desc)


# Check key type from main table
PK = '과제고유번호'
flist = sorted([x for x in os.listdir(path) if (x[0] != '.') & (x[-8:] == 'Desc.csv')])
for i, f in enumerate(tqdm(flist)):
    df_desc = pd.read_csv(PATH+f, index_col=0)
    if i == 0:
        PK_type = df_desc.loc[PK, 'Type']
    else:
        df_desc.loc[PK, 'Type'] = PK_type
        df_desc.to_csv(PATH+f)

 * Starting MySQL database server mysqld
   ...done.
Database `KR_NTIS_20240429_raw` dropped successfully.
Database `KR_NTIS_20240429_raw` created successfully.


  6%|██▋                                        | 1/16 [01:47<26:56, 107.78s/it]

Generate the Description file for table `1_Projects`


 12%|█████▌                                      | 2/16 [02:08<13:15, 56.84s/it]

Generate the Description file for table `2_Patents`


...

 94%|████████████████████████████████████████▎  | 15/16 [03:15<00:02,  2.63s/it]

Generate the Description file for table `8_TraningSupports`


100%|███████████████████████████████████████████| 16/16 [03:15<00:00, 12.22s/it]


Generate the Description file for table `9_Training`


100%|██████████████████████████████████████████| 16/16 [00:00<00:00, 556.68it/s]


In [8]:
flist = sorted([x for x in os.listdir(PATH) if '.ftr' in x])
for f in tqdm(flist[:]):
    # Generate and execute CREATE TABLE SQL
    data_config = preview.update_data_config(f, data_config)
    manage.create_table(data_config, db_config)
    manage.fill_table_from_file(data_config, db_config)
    manage.set_index(db_config, data_config)
    manage.optimize_table(db_config, data_config)

  0%|                                                    | 0/16 [00:00<?, ?it/s]

Table `1_Projects` created successfully.
Data inserted into table `1_Projects` successfully.
Set Index the `과제고유번호` on `1_Projects` successfully.


  6%|██▌                                     | 1/16 [27:17<6:49:15, 1637.02s/it]

Optimize table `1_Projects` successfully.
Table `2_Patents` created successfully.
Data inserted into table `2_Patents` successfully.


 12%|█████▏                                   | 2/16 [29:45<2:57:39, 761.37s/it]

...

Optimize table `7_Commercialize` successfully.
Table `8_TraningSupports` created successfully.
Data inserted into table `8_TraningSupports` successfully.


 94%|████████████████████████████████████████▎  | 15/16 [40:51<00:34, 34.28s/it]

Optimize table `8_TraningSupports` successfully.
Table `9_Training` created successfully.
Data inserted into table `9_Training` successfully.


100%|██████████████████████████████████████████| 16/16 [40:56<00:00, 153.53s/it]

Optimize table `9_Training` successfully.


In [10]:
data_config['table_name']

'9_Training'

In [9]:
for f in tqdm(flist[:]):
    # Generate and execute CREATE TABLE SQL
    data_config = preview.update_data_config(f, data_config)
    df_desc = preview.get_Table_Description(data_config, params)

    manage.drop_table(data_config['table_name'], db_config)
    manage.create_table(data_config, db_config)
    manage.fill_table_from_file(data_config, db_config)
    manage.set_index(db_config, data_config)
    manage.optimize_table(db_config, data_config)

  0%|                                                    | 0/16 [00:00<?, ?it/s]

Generate the Description file for table `1_Projects`
Table `1_Projects` dropped successfully.
Table `1_Projects` created successfully.
Data inserted into table `1_Projects` successfully.
Set Index the `과제고유번호` on `1_Projects` successfully.


  6%|██▌                                     | 1/16 [28:47<7:11:45, 1727.00s/it]

Optimize table `1_Projects` successfully.
Generate the Description file for table `2_Patents`
Table `2_Patents` dropped successfully.
Table `2_Patents` created successfully.
Data inserted into table `2_Patents` successfully.


 12%|█████▏                                   | 2/16 [31:32<3:08:41, 808.67s/it]

...

Optimize table `7_Commercialize` successfully.
Generate the Description file for table `8_TraningSupports`
Table `8_TraningSupports` dropped successfully.
Table `8_TraningSupports` created successfully.
Data inserted into table `8_TraningSupports` successfully.


 94%|████████████████████████████████████████▎  | 15/16 [43:51<00:37, 37.17s/it]

Optimize table `8_TraningSupports` successfully.
Generate the Description file for table `9_Training`
Table `9_Training` dropped successfully.
Table `9_Training` created successfully.
Data inserted into table `9_Training` successfully.


100%|██████████████████████████████████████████| 16/16 [43:57<00:00, 164.85s/it]

Optimize table `9_Training` successfully.


In [11]:
manage.backup_database_subprocess(db_config, data_config)

mysqldump: [Warning] Using a password on the command line interface can be insecure.


# US-NIH

In [ ]:
PATH = '../Data/Funding/US_NIH/'
SEP = '\t'
Port = 0 # Port for DB with host
CHARACTER_SET = 'utf8mb4'
COLLATE = 'utf8mb4_unicode_520_ci'

params = dict(Extra_ratio=1.5, 
              Min_Year=1900, 
              Max_Year=2100, 
              unique_ratio_th=.5, 
              freq_ratio_th=1e-3)

db_config = {
    'host': 'localhost',  # Update as needed
    'user': 'user',       # Update as needed
    'password': '1234',       # Update as needed
    'database': 'US_NIH_2023_raw'  # Update as needed
}

data_config = {
    'PATH': PATH,
    'SEP': SEP,
    'file_name': 'file_name', # Dummy init value
    'file_type': 'csv', # Dummy init value
    'table_name': 'table_name', # Dummy init value for Exporting
    'out_path': '../Data/SQL/', # Update as needed
    'Conv_DATETIME': False,
}

In [ ]:
manage.init_MySQL()
try:
    manage.drop_DB(db_config['database'], db_config)
except:
    pass
manage.create_DB(db_config['database'], CHARACTER_SET, COLLATE, db_config)


# Generate the Tabular File list
flist = sorted([x for x in os.listdir(PATH) if 'raw.ftr' in x])
for f in tqdm(flist[:]):
    data_config = preview.update_data_config(f, data_config)
    df_desc = preview.get_Table_Description(data_config, params)
    
    # Generate and execute CREATE TABLE SQL
    manage.create_table(data_config, db_config)
    manage.fill_table_from_file(data_config, db_config)
    manage.set_index(db_config, data_config)
    manage.optimize_table(db_config, data_config)

In [ ]:
manage.backup_database_subprocess(db_config, data_config)

# US_NSF

In [39]:
PATH = '../Data/Funding/US_NSF/'
SEP = '\t'
Port = 0 # Port for DB with host
CHARACTER_SET = 'utf8mb4'
COLLATE = 'utf8mb4_unicode_520_ci'

params = dict(Extra_ratio=1.5, 
              Min_Year=1900, 
              Max_Year=2100, 
              unique_ratio_th=.5, 
              freq_ratio_th=1e-3)

db_config = {
    'host': 'localhost',  # Update as needed
    'user': 'user',       # Update as needed
    'password': '1234',       # Update as needed
    'database': 'US_NSF_2023_raw'  # Update as needed
}

data_config = {
    'PATH': PATH,
    'SEP': SEP,
    'file_name': 'file_name', # Dummy init value
    'file_type': 'ftr', # Dummy init value
    'table_name': 'table_name', # Dummy init value for Exporting
    'out_path': '../Data/SQL/', # Update as needed
    'Conv_DATETIME': False,
}

In [40]:
manage.init_MySQL()
try:
    manage.drop_DB(db_config['database'], db_config)
except:
    pass
manage.create_DB(db_config['database'], CHARACTER_SET, COLLATE, db_config)


# Generate the Tabular File list
flist = sorted([x for x in os.listdir(PATH) if '.ftr' in x])
for f in tqdm(flist[:]):
    data_config = preview.update_data_config(f, data_config)
    df_desc = preview.get_Table_Description(data_config, params)
    
    # Generate and execute CREATE TABLE SQL
    manage.create_table(data_config, db_config)
    manage.fill_table_from_file(data_config, db_config)
    manage.set_index(db_config, data_config)
    manage.optimize_table(db_config, data_config)

 * Starting MySQL database server mysqld
   ...done.
Database `US_NSF_2023_raw` dropped successfully.
Database `US_NSF_2023_raw` created successfully.


  0%|                                                     | 0/9 [00:00<?, ?it/s]

Generate the Description file for table `0_Total_Award`
Table `0_Total_Award` created successfully.
Data inserted into table `0_Total_Award` successfully.
Set Index the rootTag__Award__AwardID on `0_Total_Award` successfully.


 11%|████▌                                    | 1/9 [19:02<2:32:20, 1142.52s/it]

Optimize table `0_Total_Award` successfully.
Generate the Description file for table `1_Total_Institution`
Table `1_Total_Institution` created successfully.
Data inserted into table `1_Total_Institution` successfully.
Set Index the rootTag__Award__AwardID on `1_Total_Institution` successfully.


...

Optimize table `7_Total_ProgramReference` successfully.
Generate the Description file for table `8_Total_Appropriation`
Table `8_Total_Appropriation` created successfully.
Data inserted into table `8_Total_Appropriation` successfully.
Set Index the rootTag__Award__AwardID on `8_Total_Appropriation` successfully.


 89%|████████████████████████████████████████     | 8/9 [24:44<01:11, 71.17s/it]

Optimize table `8_Total_Appropriation` successfully.
Generate the Description file for table `9_Total_Fund`
Table `9_Total_Fund` created successfully.
Data inserted into table `9_Total_Fund` successfully.
Set Index the rootTag__Award__AwardID on `9_Total_Fund` successfully.


100%|████████████████████████████████████████████| 9/9 [25:10<00:00, 167.83s/it]

Optimize table `9_Total_Fund` successfully.


In [41]:
manage.backup_database_subprocess(db_config, data_config)

mysqldump: [Warning] Using a password on the command line interface can be insecure.


# Verbose

In [23]:
df = preview.read_data_from_tabular(data_config)

In [248]:
df

,rootTag.Award.AwardID,rootTag.Award.AGENCY,rootTag.Award.ARRAAmount,rootTag.Award.AWDG_AGCY_CODE,rootTag.Award.AbstractNarration,rootTag.Award.AwardAmount,rootTag.Award.AwardEffectiveDate,rootTag.Award.AwardExpirationDate,rootTag.Award.AwardInstrument.Value,rootTag.Award.AwardTitle,...,rootTag.Award.Organization.Directorate.Abbreviation,rootTag.Award.Organization.Directorate.LongName,rootTag.Award.Organization.Division.Abbreviation,rootTag.Award.Organization.Division.LongName,rootTag.Award.ProgramOfficer.PO_EMAI,rootTag.Award.ProgramOfficer.PO_PHON,rootTag.Award.ProgramOfficer.SignBlockName,rootTag.Award.TRAN_TYPE,rootTag.Award.POR.DRECONTENT,rootTag.Award.POR.POR_COPY_TXT
0,5900035,NSF,None,4900,None,4000000,1967-06-30,1969-09-30,Contract,Contract For Construction and Operation of the...,...,MPS,Direct For Mathematical & Physical Scien,AST,Division Of Astronomical Sciences,None,None,name not available,Grant,None,None
1,5900036,NSF,None,4900,None,None,1967-06-30,1971-12-31,Contract,Contract For Bid on Preparation of Reports on ...,...,CSE,Direct For Computer & Info Scie & Enginr,IIS,Div Of Information & Intelligent Systems,None,None,Oscar Garcia,Grant,None,None
2,5900037,NSF,None,4900,None,291000,1967-06-30,1969-02-28,Contract-BOA/Task Order,Task Order For the Pacific Science Board,...,O/D,Office Of The Director,OISE,Office Of Internatl Science &Engineering,None,None,name not available,Grant,None,None
3,5900038,NSF,None,4900,None,370030,1967-06-30,1972-06-30,Contract-BOA/Task Order,Contract For Committee on Educational Policies...,...,CSE,Direct For Computer & Info Scie & Enginr,IIS,Div Of Information & Intelligent Systems,None,None,name not available,Grant,None,None
4,5900039,NSF,None,4900,None,345000,1967-06-30,1970-06-30,Contract-BOA/Task Order,Task Order For Support of a Committee on ...,...,CSE,Direct For Computer & Info Scie & Enginr,IIS,Div Of Information & Intelligent Systems,None,None,name not available,Grant,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512660,2350377,NSF,None,4900,Researchers conducting lab-based human subject...,16029,2023-10-01,2027-03-31,Continuing Grant,Collaborative Research: CCRI: Grand: Virtual E...,...,CSE,Direct For Computer & Info Scie & Enginr,CNS,Division Of Computer and Network Systems,eglinert@nsf.gov,7032924341,Ephraim Glinert,Grant,None,None
512661,2350392,NSF,None,4900,This project will shed light on how a DNA-dire...,323335,2023-10-01,2025-03-31,Continuing Grant,Structural insights into RNA promoters for RNA...,...,BIO,Direct For Biological Sciences,MCB,Div Of Molecular and Cellular Bioscience,amushegi@nsf.gov,7032928528,Arcady Mushegian,Grant,None,None
512662,2350473,NSF,None,4900,Biological collections are a primary source of...,339625,2023-10-01,2025-12-31,Standard Grant,CSBR: Emergency Rescue of the Orphaned Fish Co...,...,BIO,Direct For Biological Sciences,DBI,Div Of Biological Infrastructure,mherron@nsf.gov,7032925361,Matthew Herron,Grant,None,None
512663,2350521,NSF,None,4900,This award is funded in whole or in part under...,183252,2023-10-01,2024-03-31,Standard Grant,ERI: CAS- Climate: Design for Recyclability: P...,...,ENG,Directorate For Engineering,CBET,"Div Of Chem, Bioeng, Env, & Transp Sys",bhamilto@nsf.gov,7032920000,Bruce Hamilton,Grant,None,None


In [251]:
import pymysql

from KISTI_DB_Manager import manage, preview
import importlib as imp
imp.reload(manage), imp.reload(preview)

(<module 'KISTI_DB_Manager.manage' from '/workspace/KISTI_DB_Manager/KISTI_DB_Manager/manage.py'>,
 <module 'KISTI_DB_Manager.preview' from '/workspace/KISTI_DB_Manager/KISTI_DB_Manager/preview.py'>)

In [242]:
df_desc = preview.get_Table_Description(data_config, params, True)

rootTag.Award.AwardID
rootTag.Award.AGENCY
rootTag.Award.ARRAAmount
rootTag.Award.AWDG_AGCY_CODE
rootTag.Award.AbstractNarration
rootTag.Award.AwardAmount
rootTag.Award.AwardEffectiveDate
rootTag.Award.AwardExpirationDate
rootTag.Award.AwardInstrument.Value
rootTag.Award.AwardTitle
rootTag.Award.AwardTotalIntnAmount
rootTag.Award.CFDA_NUM
rootTag.Award.FUND_AGCY_CODE
rootTag.Award.MaxAmdLetterDate
rootTag.Award.MinAmdLetterDate
rootTag.Award.NSF_PAR_USE_FLAG
rootTag.Award.Organization.Code
rootTag.Award.Organization.Directorate.Abbreviation
rootTag.Award.Organization.Directorate.LongName
rootTag.Award.Organization.Division.Abbreviation
rootTag.Award.Organization.Division.LongName
rootTag.Award.ProgramOfficer.PO_EMAI
rootTag.Award.ProgramOfficer.PO_PHON
rootTag.Award.ProgramOfficer.SignBlockName
rootTag.Award.TRAN_TYPE
rootTag.Award.POR.DRECONTENT
rootTag.Award.POR.POR_COPY_TXT
Generate the Description file for table `0_Total_Award`


In [243]:
df_desc
# df_desc.index = [x.replace('.', '_') for x in df_desc.index]
# df_desc

,Description,Type,Example,Coverage,min,max,mean,std,top,uniq,freq,entr,Failed,_max_range,uniq_ratio,Null_ratio,is_key
rootTag_Award_AwardID,,MEDIUMINT UNSIGNED,5900035,1.0,0,9996454,NaN,NaN,1022648,512660,0.000004,18.967637,False,16777214,0.99999,0.01,True
rootTag_Award_AGENCY,,VARCHAR(4),NSF,0.875028,3,3,NaN,NaN,NSF,2,0.875028,0.16853,False,4,0.000004,0.01,False
rootTag_Award_ARRAAmount,,INT UNSIGNED,18541667,0.010055,2273,148070000,NaN,NaN,400000,4005,0.989945,0.18053,False,4294967294,0.007812,0.01,False
rootTag_Award_AWDG_AGCY_CODE,,SMALLINT UNSIGNED,4900,0.875028,4900,4900,NaN,NaN,4900,2,0.875028,0.16853,False,65534,0.000004,0.01,False
rootTag_Award_AbstractNarration,,TEXT,"Due to their proven technology, reciprocating-...",0.783083,1,10000,NaN,NaN,Presidential Awards for Excellence in Science ...,363816,0.216917,14.66853,False,15000,0.709656,0.01,False
rootTag_Award_AwardAmount,,BIGINT,4000000,0.999052,-47492,1826832571,NaN,NaN,50000,230253,0.012462,15.315891,False,9223372036854775807,0.44913,0.01,False
rootTag_Award_AwardEffectiveDate,,DATETIME,06/30/1967,1.0,1900-01-01 00:00:00,2024-11-01 00:00:00,NaN,NaN,09/01/2009,5834,0.007482,9.887129,False,-,0.01138,0.01,False
rootTag_Award_AwardExpirationDate,,DATETIME,09/30/1969,1.0,1900-01-01 00:00:00,2032-09-30 00:00:00,NaN,NaN,08/31/2024,2752,0.010022,9.160331,False,-,0.005368,0.01,False
rootTag_Award_AwardInstrument_Value,,VARCHAR(64),Contract,1.0,3,33,NaN,NaN,Standard Grant,16,0.658939,1.381798,False,49,0.000031,0.01,False
rootTag_Award_AwardTitle,,TEXT,Contract For Construction and Operation of the...,0.999945,1,180,NaN,NaN,Instructional Scientific Equipment Program,422604,0.002528,18.30913,False,270,0.824328,0.01,False


In [176]:
df = preview.read_data_from_tabular(data_config)
col = 'rootTag.Award.AwardEffectiveDate'
_series = df[col]
preview.get_Field_Description(_series)

Description                       
Type                      DATETIME
Example                 06/30/1967
Coverage                       1.0
min            1900-01-01 00:00:00
max            2024-11-01 00:00:00
mean                           NaN
std                            NaN
top                     09/01/2009
uniq                          5834
freq                      0.007482
entr                      9.887129
Failed                       False
_max_range                       -
uniq_ratio                 0.01138
Null_ratio                    0.01
is_key                       False
Name: attributes, dtype: object

In [178]:
_series = df[col]
preview.get_Field_Description(_series)

Description                       
Type                      DATETIME
Example                 06/30/1967
Coverage                       1.0
min            1900-01-01 00:00:00
max            2024-11-01 00:00:00
mean                           NaN
std                            NaN
top                     09/01/2009
uniq                          5834
freq                      0.007482
entr                      9.887129
Failed                       False
_max_range                       -
uniq_ratio                 0.01138
Null_ratio                    0.01
is_key                       False
Name: attributes, dtype: object

In [177]:
preview.get_MariaDB_Type(_series.dropna())

Type                     DATETIME
min           1900-01-01 00:00:00
max           2024-11-01 00:00:00
_max_range                      -
Failed                      False
dtype: object

In [211]:
msk = df_desc.Type == 'DATETIME'
cols = [x for x in df_desc[msk].index]
cols = df_desc[msk].index

col = cols[0]

In [216]:
pd.to_datetime(df[col]).dt.strftime('%Y-%m-%d %H:%M:%S')

0         1967-06-30 00:00:00
1         1967-06-30 00:00:00
2         1967-06-30 00:00:00
3         1967-06-30 00:00:00
4         1967-06-30 00:00:00
                 ...         
512660    2023-10-01 00:00:00
512661    2023-10-01 00:00:00
512662    2023-10-01 00:00:00
512663    2023-10-01 00:00:00
512664    2023-10-01 00:00:00
Name: rootTag.Award.AwardEffectiveDate, Length: 512665, dtype: object

In [196]:
_series = df['rootTag.Award.AwardEffectiveDate']
pd.to_datetime(_series)

0        1967-06-30
1        1967-06-30
2        1967-06-30
3        1967-06-30
4        1967-06-30
            ...    
512660   2023-10-01
512661   2023-10-01
512662   2023-10-01
512663   2023-10-01
512664   2023-10-01
Name: rootTag.Award.AwardEffectiveDate, Length: 512665, dtype: datetime64[ns]

In [197]:
df_desc

,Description,Type,Example,Coverage,min,max,mean,std,top,uniq,freq,entr,Failed,_max_range,uniq_ratio,Null_ratio,is_key
rootTag.Award.AwardID,,MEDIUMINT UNSIGNED,5900035,1.0,0,9996454,NaN,NaN,1022648,512660,0.000004,18.967637,False,16777214,0.99999,0.01,True
rootTag.Award.AGENCY,,VARCHAR(4),NSF,0.875028,3,3,NaN,NaN,NSF,2,0.875028,0.16853,False,4,0.000004,0.01,False
rootTag.Award.ARRAAmount,,INT UNSIGNED,18541667,0.010055,2273,148070000,NaN,NaN,400000,4005,0.989945,0.18053,False,4294967294,0.007812,0.01,False
rootTag.Award.AWDG_AGCY_CODE,,SMALLINT UNSIGNED,4900,0.875028,4900,4900,NaN,NaN,4900,2,0.875028,0.16853,False,65534,0.000004,0.01,False
rootTag.Award.AbstractNarration,,TEXT,"Due to their proven technology, reciprocating-...",0.783083,1,10000,NaN,NaN,Presidential Awards for Excellence in Science ...,363816,0.216917,14.66853,False,15000,0.709656,0.01,False
rootTag.Award.AwardAmount,,BIGINT,4000000,0.999052,-47492,1826832571,NaN,NaN,50000,230253,0.012462,15.315891,False,9223372036854775807,0.44913,0.01,False
rootTag.Award.AwardEffectiveDate,,DATETIME,06/30/1967,1.0,1900-01-01 00:00:00,2024-11-01 00:00:00,NaN,NaN,09/01/2009,5834,0.007482,9.887129,False,-,0.01138,0.01,False
rootTag.Award.AwardExpirationDate,,DATETIME,09/30/1969,1.0,1900-01-01 00:00:00,2032-09-30 00:00:00,NaN,NaN,08/31/2024,2752,0.010022,9.160331,False,-,0.005368,0.01,False
rootTag.Award.AwardInstrument.Value,,VARCHAR(64),Contract,1.0,3,33,NaN,NaN,Standard Grant,16,0.658939,1.381798,False,49,0.000031,0.01,False
rootTag.Award.AwardTitle,,TEXT,Contract For Construction and Operation of the...,0.999945,1,180,NaN,NaN,Instructional Scientific Equipment Program,422604,0.002528,18.30913,False,270,0.824328,0.01,False


In [230]:
manage.drop_table(data_config['table_name'], db_config)
manage.create_table(data_config, db_config)
manage.fill_table_from_file(data_config, db_config)
    

Data inserted into table `0_Total_Award` successfully.


In [233]:
manage.set_index(db_config, data_config)
# manage.optimize_table(db_config, data_config)

Failed to create index `0_Total_Award`. Error: (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '.Award.AwardID)' at line 1") with CREATE INDEX `IDX_0_TOTAL_AWARD_ROOTTAG.AWARD.AWARDID` ON `0_Total_Award` (rootTag.Award.AwardID);


In [234]:
data_config

{'PATH': '../Data/Funding/US_NSF/',
 'SEP': '\t',
 'file_name': '0_Total_Award.ftr',
 'file_type': 'ftr',
 'table_name': '0_Total_Award',
 'out_path': '../Data/SQL/'}